In [1]:
import numpy as np



# 1 Define the haptic actions


HAPTIC_ACTIONS = {
    0: "Low",
    1: "Medium",
    2: "High"
}

print("Haptic actions:")
print(HAPTIC_ACTIONS)



# 2 Define the state-dependent haptic effect


def haptic_effect(
    action,
    difficulty,
    cognitive_effort,
    performance
):
    need = (
        0.4 * difficulty
        + 0.4 * cognitive_effort
        + 0.2 * (1 - performance)
    )

    if action == 0:
        effect = 0.00

    elif action == 1:
        effect = 0.15 * need

    elif action == 2:
        effect = 0.30 * need

    else:
        raise ValueError(
            "Action must be 0, 1, or 2."
        )

    return effect



# 3 Calculate response probability without haptic feedback


def probability_correct(
    ability,
    difficulty,
    cognitive_effort,
    alpha=4.0,
    beta=3.0,
    gamma=2.0
):
    score = (
        alpha * ability
        - beta * difficulty
        - gamma * cognitive_effort
    )

    return (
        1
        / (
            1
            + np.exp(-score)
        )
    )



# 4 Calculate response probability with haptic feedback


def probability_correct_with_haptics(
    ability,
    difficulty,
    cognitive_effort,
    performance,
    action,
    alpha=4.0,
    beta=3.0,
    gamma=2.0
):
    base_score = (
        alpha * ability
        - beta * difficulty
        - gamma * cognitive_effort
    )

    effect = haptic_effect(
        action=action,
        difficulty=difficulty,
        cognitive_effort=cognitive_effort,
        performance=performance
    )

    final_score = (
        base_score
        + effect
    )

    return (
        1
        / (
            1
            + np.exp(-final_score)
        )
    )



# 5 Define the haptic-action costs


HAPTIC_COST = {
    0: 0.00,
    1: 0.03,
    2: 0.08
}

print("\nHaptic costs:")
print(HAPTIC_COST)



# 6 Define the reward function


def calculate_reward(
    correct,
    p_before,
    p_after,
    action
):
    # Reward a successful response and penalise an incorrect response
    if correct == 1:
        success_reward = 1.0

    else:
        success_reward = -0.5

    # Improvement in response probability after haptic support
    improvement = (
        p_after
        - p_before
    )

    # Cost associated with intervention strength
    haptic_cost = (
        HAPTIC_COST[action]
    )

    # Final reward
    reward = (
        success_reward
        + 2.0 * improvement
        - haptic_cost
    )

    return reward



# 7 Check the reward function on example outcomes


reward_examples = [
    {
        "name": "Low need + Low haptic",
        "correct": 1,
        "p_before": 0.80,
        "p_after": 0.80,
        "action": 0
    },

    {
        "name": "Low need + High haptic",
        "correct": 1,
        "p_before": 0.80,
        "p_after": 0.82,
        "action": 2
    },

    {
        "name": "High need + High haptic",
        "correct": 1,
        "p_before": 0.20,
        "p_after": 0.32,
        "action": 2
    },

    {
        "name": "High need + Low haptic",
        "correct": 0,
        "p_before": 0.20,
        "p_after": 0.20,
        "action": 0
    }
]

print(
    "\nReward sanity check:"
)

for example in reward_examples:

    reward = calculate_reward(
        correct=example["correct"],
        p_before=example["p_before"],
        p_after=example["p_after"],
        action=example["action"]
    )

    print(
        example["name"],
        "-> Reward:",
        round(
            reward,
            3
        )
    )



# 8 Define one learner-environment interaction


def environment_step(
    row,
    performance,
    action,
    rng
):
    ability = float(
        row[
            "Learner_Ability"
        ]
    )

    difficulty = float(
        row[
            "Difficulty_D"
        ]
    )

    cognitive_effort = float(
        row[
            "Cognitive_Effort_C"
        ]
    )

    # Baseline response probability
    p_before = probability_correct(
        ability=ability,
        difficulty=difficulty,
        cognitive_effort=cognitive_effort
    )

    # Response probability after selected haptic action
    p_after = probability_correct_with_haptics(
        ability=ability,
        difficulty=difficulty,
        cognitive_effort=cognitive_effort,
        performance=performance,
        action=action
    )

    # Numerical safety
    p_after = np.clip(
        p_after,
        0.0,
        1.0
    )

    # Sample the learner response
    correct = rng.binomial(
        n=1,
        p=p_after
    )

    # Calculate reward
    reward = calculate_reward(
        correct=correct,
        p_before=p_before,
        p_after=p_after,
        action=action
    )

    # Dynamically update learner performance
    new_performance = (
        0.8 * performance
        + 0.2 * correct
    )

    new_performance = np.clip(
        new_performance,
        0.0,
        1.0
    )

    return (
        new_performance,
        reward,
        correct,
        p_before,
        p_after
    )



# 9 Check deterministic environment quantities


test_row = {
    "Learner_Ability": 0.55,
    "Difficulty_D": 0.60,
    "Cognitive_Effort_C": 0.70
}

performance = 0.50

print(
    "\nEnvironment probability check:"
)

for action in HAPTIC_ACTIONS:

    p_before = probability_correct(
        ability=test_row[
            "Learner_Ability"
        ],
        difficulty=test_row[
            "Difficulty_D"
        ],
        cognitive_effort=test_row[
            "Cognitive_Effort_C"
        ]
    )

    p_after = probability_correct_with_haptics(
        ability=test_row[
            "Learner_Ability"
        ],
        difficulty=test_row[
            "Difficulty_D"
        ],
        cognitive_effort=test_row[
            "Cognitive_Effort_C"
        ],
        performance=performance,
        action=action
    )

    effect = haptic_effect(
        action=action,
        difficulty=test_row[
            "Difficulty_D"
        ],
        cognitive_effort=test_row[
            "Cognitive_Effort_C"
        ],
        performance=performance
    )

    print(
        "\nAction:",
        HAPTIC_ACTIONS[action]
    )

    print(
        "Haptic effect:",
        round(
            effect,
            4
        )
    )

    print(
        "P before:",
        round(
            p_before,
            4
        )
    )

    print(
        "P after:",
        round(
            p_after,
            4
        )
    )

    print(
        "Probability change:",
        round(
            p_after
            - p_before,
            4
        )
    )



# 10 Check the complete stochastic environment step


print(
    "\nComplete environment-step check:"
)

for action in HAPTIC_ACTIONS:

    # Reset the random generator so each action begins
    # from the same reproducible random state
    rng = np.random.default_rng(
        42
    )

    (
        new_performance,
        reward,
        correct,
        p_before,
        p_after
    ) = environment_step(
        row=test_row,
        performance=performance,
        action=action,
        rng=rng
    )

    print(
        "\nAction:",
        HAPTIC_ACTIONS[action]
    )

    print(
        "P before:",
        round(
            p_before,
            4
        )
    )

    print(
        "P after:",
        round(
            p_after,
            4
        )
    )

    print(
        "Correct:",
        correct
    )

    print(
        "Reward:",
        round(
            reward,
            4
        )
    )

    print(
        "New performance:",
        round(
            new_performance,
            4
        )
    )



# 11 Check performance-update behaviour


example_performance = 0.50

performance_after_correct = (
    0.8 * example_performance
    + 0.2 * 1
)

performance_after_incorrect = (
    0.8 * example_performance
    + 0.2 * 0
)

print(
    "\nPerformance update check:"
)

print(
    "Starting performance:",
    example_performance
)

print(
    "After correct response:",
    performance_after_correct
)

print(
    "After incorrect response:",
    performance_after_incorrect
)



# 12 Confirm Phase 13 completion


print(
    "\nPHASE 13 COMPLETE"
)

print(
    "Reward and environment functions "
    "are ready for Q-learning."
)

Haptic actions:
{0: 'Low', 1: 'Medium', 2: 'High'}

Haptic costs:
{0: 0.0, 1: 0.03, 2: 0.08}

Reward sanity check:
Low need + Low haptic -> Reward: 1.0
Low need + High haptic -> Reward: 0.96
High need + High haptic -> Reward: 1.16
High need + Low haptic -> Reward: -0.5

Environment probability check:

Action: Low
Haptic effect: 0.0
P before: 0.2689
P after: 0.2689
Probability change: 0.0

Action: Medium
Haptic effect: 0.093
P before: 0.2689
P after: 0.2876
Probability change: 0.0187

Action: High
Haptic effect: 0.186
P before: 0.2689
P after: 0.307
Probability change: 0.0381

Complete environment-step check:

Action: Low
P before: 0.2689
P after: 0.2689
Correct: 1
Reward: 1.0
New performance: 0.6

Action: Medium
P before: 0.2689
P after: 0.2876
Correct: 1
Reward: 1.0073
New performance: 0.6

Action: High
P before: 0.2689
P after: 0.307
Correct: 1
Reward: 0.9962
New performance: 0.6

Performance update check:
Starting performance: 0.5
After correct response: 0.6000000000000001
After inc